# Mohamed Soueidatt - C12989

#### Solutions des exercices du chapitre 19

### Ex. 1 - Addition de polynomes

In [1]:
import numpy as np

class Polynomial:
    def __init__(self, coeff=None, points=None, basis='monomial'):
        self.basis = basis
        self.points = points or []
        self.coeff = np.array(coeff if coeff is not None else [], dtype=float)
        self.degree = len(self.coeff) - 1

    @staticmethod
    def from_points(points):
        x = np.array([p[0] for p in points], dtype=float)
        y = np.array([p[1] for p in points], dtype=float)
        coeff = np.polyfit(x, y, len(points)-1)[::-1]
        return Polynomial(coeff=coeff, points=list(points), basis='monomial')

    def __call__(self, x):
        if self.basis == 'monomial':
            return sum(c*x**i for i, c in enumerate(self.coeff))
        result = self.coeff[-1]
        abscissas = [p[0] for p in self.points]
        for c, a in zip(self.coeff[-2::-1], abscissas[-2::-1]):
            result = result*(x - a) + c
        return result

    def __changepoints__(self, new_points):
        values = [(x, self(x)) for x, _ in new_points]
        return Polynomial.from_points(values).coeff

    def __add__(self, other):
        p = self.to_monomial()
        q = other.to_monomial()
        n = max(len(p.coeff), len(q.coeff))
        a = np.pad(p.coeff, (0, n-len(p.coeff)))
        b = np.pad(q.coeff, (0, n-len(q.coeff)))
        return Polynomial(coeff=a+b, basis='monomial')

    def to_monomial(self):
        if self.basis == 'monomial':
            return self
        return Polynomial.from_points([(x, self(x)) for x, _ in self.points])

p = Polynomial(coeff=[1, 2, 1])       # 1 + 2x + x^2
q = Polynomial(coeff=[0, -1, 3])      # -x + 3x^2
r = p + q
r.coeff.tolist(), r(2)

([1.0, 1.0, 4.0], 19.0)

### Ex. 2 - Conversion entre formes monomiale et Newton

In [2]:
def divided_differences(points):
    x = np.array([p[0] for p in points], dtype=float)
    a = np.array([p[1] for p in points], dtype=float)
    n = len(points)
    for j in range(1, n):
        a[j:n] = (a[j:n] - a[j-1:n-1])/(x[j:n] - x[:n-j])
    return a

class NewtonPolynomial(Polynomial):
    @staticmethod
    def from_points(points):
        return NewtonPolynomial(coeff=divided_differences(points), points=list(points), basis='newton')

points = [(0, 1), (1, 4), (2, 9)]
newton = NewtonPolynomial.from_points(points)
mono = newton.to_monomial()
newton.coeff.tolist(), [round(c, 8) for c in mono.coeff]

([1.0, 3.0, 1.0], [1.0, 2.0, 1.0])

### Ex. 3 - Methode `add_point`

In [3]:
def add_point(poly, point):
    new_points = list(poly.points) + [point]
    return Polynomial.from_points(new_points)

p = Polynomial.from_points([(0, 1), (1, 4), (2, 9)])
p2 = add_point(p, (3, 16))
[p2(x) for x in [0, 1, 2, 3]], [round(c, 8) for c in p2.coeff]

([1.000000000000003, 4.000000000000002, 9.000000000000002, 16.000000000000004], [1.0, 2.0, 1.0, -0.0])

### Ex. 4 - Classe `LagrangePolynomial`

In [4]:
class LagrangePolynomial(Polynomial):
    def __init__(self, points):
        super().__init__(coeff=None, points=list(points), basis='lagrange')
        self.degree = len(points) - 1

    def __call__(self, x):
        total = 0.0
        for i, (xi, yi) in enumerate(self.points):
            Li = 1.0
            for j, (xj, _) in enumerate(self.points):
                if i != j:
                    Li *= (x - xj)/(xi - xj)
            total += yi*Li
        return total

lag = LagrangePolynomial([(0, 1), (1, 4), (2, 9)])
[lag(x) for x in [0, 0.5, 1, 2]]

[1.0, 2.25, 4.0, 9.0]

### Ex. 5 - Tests de la classe polynomiale

In [5]:
def test_polynomials():
    p = Polynomial(coeff=[1, 2, 1])
    assert abs(p(3) - 16) < 1e-12

    q = Polynomial.from_points([(0, 1), (1, 4), (2, 9)])
    assert np.allclose([q(0), q(1), q(2)], [1, 4, 9])

    n = NewtonPolynomial.from_points([(0, 1), (1, 4), (2, 9)])
    assert abs(n(2) - 9) < 1e-12

    l = LagrangePolynomial([(0, 1), (1, 4), (2, 9)])
    assert abs(l(0.5) - 2.25) < 1e-12

    assert abs((p + p)(2) - 18) < 1e-12
    return 'Tous les tests passent.'

test_polynomials()

'Tous les tests passent.'